In [1]:
import xml.etree.ElementTree as ET
from shapely import Point
from shapely import Polygon

In [ ]:
def file_coords(xml_file_path = r'E:\ISRO PS-Transformer\DataSet\OHRC\ch2_ohr_nrp_20200229T0938004033_d_img_d32\data\raw\20200229\ch2_ohr_nrp_20200229T0938004033_d_img_d32.xml'):
    tree = ET.parse(xml_file_path)
    root = tree.getroot()

    # Define the namespace dictionary
    namespace = {
        'isda': 'https://isda.issdc.gov.in/pds4/isda/v1',
    }

    # Function to extract latitude and longitude values
    def extract_coordinates(element):
        coordinates = element.find('.//isda:System_Level_Coordinates', namespace)
        upper_left_latitude = coordinates.find('./isda:upper_left_latitude', namespace).text
        upper_left_longitude = coordinates.find('./isda:upper_left_longitude', namespace).text

        upper_right_latitude = coordinates.find('./isda:upper_right_latitude', namespace).text
        upper_right_longitude = coordinates.find('./isda:upper_right_longitude', namespace).text

        lower_left_latitude = coordinates.find('./isda:lower_left_latitude', namespace).text
        lower_left_longitude = coordinates.find('./isda:lower_left_longitude', namespace).text

        lower_right_latitude = coordinates.find('./isda:lower_right_latitude', namespace).text
        lower_right_longitude = coordinates.find('./isda:lower_right_longitude', namespace).text

        return [(upper_left_latitude, upper_left_longitude), (upper_right_latitude, upper_right_longitude), (lower_left_latitude, lower_left_longitude), (lower_right_latitude, lower_right_longitude)]

    # Extract coordinates from the XML
    coordinates = extract_coordinates(root)
    # Print the results
    return coordinates

In [ ]:
print(Polygon(file_coords()))

POLYGON ((-73.905831 42.753003, -73.897901 42.413607, -73.064343 42.9873, -73.056822 42.664643, -73.905831 42.753003))


In [ ]:
# import imageio

# def convert_large_img_to_png(input_img_path, output_png_path):
#     try:
#         # Open the large image using imageio
#         with imageio.get_reader(input_img_path) as img_reader:
#             # Write the image in chunks to the output PNG file
#             with imageio.get_writer(output_png_path, format='PNG') as img_writer:
#                 for frame in img_reader:
#                     img_writer.append_data(frame)

#         print(f"Image saved as {output_png_path}")
#     except Exception as e:
#         print(f"Error: {e}")

# # Example usage
# input_img_path = r"E:\ISRO PS-Transformer\DataSet\OHRC\ch2_ohr_nrp_20200229T0938004033_d_img_d32\data\raw\20200229\ch2_ohr_nrp_20200229T0938004033_d_img_d32.img"  # Replace with your large .img file path
# output_png_path = "output_image.png"

# convert_large_img_to_png(input_img_path, output_png_path)


In [ ]:
# import subprocess

# def convert_large_img_to_png_with_gdal(input_img_path, output_png_path):
#     try:
#         # Run gdal_translate command to convert .img to .png
#         subprocess.run(['gdal_translate', input_img_path, output_png_path])
#         print(f"Image saved as {output_png_path}")
#     except Exception as e:
#         print(f"Error: {e}")

# # Example usage
# input_img_path = r"E:\ISRO PS-Transformer\DataSet\OHRC\ch2_ohr_nrp_20200229T0938004033_d_img_d32\data\raw\20200229\ch2_ohr_nrp_20200229T0938004033_d_img_d32.img"
# output_png_path = "output_image.png"

# convert_large_img_to_png_with_gdal(input_img_path, output_png_path)


In [ ]:
from shapely.geometry import Polygon
from shapely.ops import unary_union

def common_intersec(fig1, fig2):
    # Create two polygons
    polygon1 = Polygon(fig1)
    polygon2 = Polygon(fig2)

    # Find the intersection (common area) between the polygons
    intersection = polygon1.intersection(polygon2)

    # If the intersection is a Polygon (not empty), get its exterior (boundary) coordinates
    if intersection.is_empty:
        return None
    else:
        if intersection.geom_type == 'Polygon':
            common_area_coordinates = list(intersection.exterior.coords)
            return common_area_coordinates
        else:
            # If the intersection is a MultiPolygon, consider the first polygon's exterior
            first_polygon = list(intersection)[0]
            common_area_coordinates = list(first_polygon.exterior.coords)
            return common_area_coordinates


In [ ]:
img1 = file_coords(r'E:\ISRO PS-Transformer\DataSet\OHRC\ch2_ohr_ncp_20230818T1143415582_d_img_n18\data\calibrated\20230818\ch2_ohr_ncp_20230818T1143415582_d_img_n18.xml')
img2 = file_coords(r'E:\ISRO PS-Transformer\DataSet\TMC\ch2_tmc_ndn_20230223T1106344149_d_oth_n18\data\derived\20230223\ch2_tmc_ndn_20230223T1106344149_d_oth_n18.xml')

In [ ]:
print(img1, img2)

[('-72.235275', '100.212078'), ('-72.234792', '100.531295'), ('-73.072298', '100.217497'), ('-73.071792', '100.551661')] [('-61.241288', '79.054394'), ('-61.233373', '77.851192'), ('-37.036545', '79.179163'), ('-37.032250', '78.470821')]


In [ ]:
common_intersec(img1, img2)

In [ ]:
import cv2
import numpy as np

def cut_out_quadrilateral(image_path, output_path, src_coordinates, dst_coordinates):
    # Read the image
    image = cv2.imread(image_path)

    # Convert the coordinates to NumPy arrays
    src_pts = np.array(src_coordinates, dtype=np.float32)
    dst_pts = np.array(dst_coordinates, dtype=np.float32)

    # Calculate the perspective transformation matrix
    perspective_matrix = cv2.getPerspectiveTransform(src_pts, dst_pts)

    # Apply the perspective transformation to obtain the cut-out region
    warped_image = cv2.warpPerspective(image, perspective_matrix, (image.shape[1], image.shape[0]))

    # Save the cut-out region to a new image file
    cv2.imwrite(output_path, warped_image)

if __name__ == "__main__":
    # Replace these values with your actual image path and output path
    input_image_path = "path/to/your/image.jpg"
    output_image_path = "path/to/save/cutout_region.jpg"

    # Source coordinates (coordinates from the original image)
    src_coordinates = [(77.65, 72.24), (60.54, 62.56), (33.65, 32.24), (24.54, 22.56)]

    # Destination coordinates (coordinates for the cut-out region)
    dst_coordinates = [(57.65, 52.24), (40.54, 42.56), (37.65, 32.24), (34.54, 30.56)]

    cut_out_quadrilateral(input_image_path, output_image_path, src_coordinates, dst_coordinates)


AttributeError: 'NoneType' object has no attribute 'shape'

In [ ]:
print(len(Polygon(img1).exterior.coords))

5


In [ ]:
files1={}
files2={}

In [ ]:
iterator = 0
for image_xml_tmc in files1:
    for image_xml_ohrc in files2:
        intersect = common_intersec(files1[image_xml_tmc], files2[image_xml_ohrc])
        if(intersect == None and len(Polygon(intersect).exterior.coords)==5):
            cut_out_quadrilateral(files1, output_path1 + '/iterator', files1[image_xml_tmc], intersect)
            cut_out_quadrilateral(files1, output_path2 + '/iterator', files1[image_xml_tmc], intersect)


In [ ]:
import os

def browse_folders(base_dir):
    for entry in os.scandir(base_dir):
        if entry.is_dir():
            folder_name = entry.name
            print(folder_name+rf"\data\calibrated\{folder_name[12:20]}\{folder_name}.xml")  # Replace this line with your processing logic

# Specify the base directory
base_directory = r'E:\ISRO PS-Transformer\DataSet\OHRC'

# Call the function with your specified base directory
browse_folders(base_directory)


ch2_ohr_ncp_20230818T1143415582_d_img_n18\data\calibrated\20230818\ch2_ohr_ncp_20230818T1143415582_d_img_n18.xml
ch2_ohr_nrp_20200229T0938004033_d_img_d32\data\calibrated\20200229\ch2_ohr_nrp_20200229T0938004033_d_img_d32.xml


In [ ]:
files1={}
files2={}

def iterate_files(base_dir):
    for root, dirs, files in os.walk(base_dir):
        for file in files:
            if file.endswith('.xml'):
                file_path = os.path.join(root, file)
                return file_path

def browse_folders(base_dir, data):
    for entry in os.scandir(base_dir):
        if entry.is_dir():
            folder_name = entry.name
            data[folder_name] = file_coords(iterate_files(os.path.join(base_dir, folder_name, "data")))


# Specify the base directory
base_directory1 = r'E:\ISRO PS-Transformer\DataSet\TMC'
base_directory2 = r'E:\ISRO PS-Transformer\DataSet\OHRC'

browse_folders(base_directory1, files1)
browse_folders(base_directory2, files2)


In [ ]:
print(files1, files2)

{'ch2_tmc_ndn_20230223T1106344149_d_oth_n18': [('-61.241288', '79.054394'), ('-61.233373', '77.851192'), ('-37.036545', '79.179163'), ('-37.032250', '78.470821')]} {'ch2_ohr_ncp_20230818T1143415582_d_img_n18': [('-72.235275', '100.212078'), ('-72.234792', '100.531295'), ('-73.072298', '100.217497'), ('-73.071792', '100.551661')], 'ch2_ohr_nrp_20200229T0938004033_d_img_d32': [('-73.905831', '42.753003'), ('-73.897901', '42.413607'), ('-73.064343', '42.987300'), ('-73.056822', '42.664643')]}


In [ ]:
import os

base_directory = r'E:\ISRO PS-Transformer\DataSet\OHRC'



# Call the function with your specified base directory
iterate_files(base_directory)


E:\ISRO PS-Transformer\DataSet\OHRC\ch2_ohr_ncp_20230818T1143415582_d_img_n18\browse\calibrated\20230818\ch2_ohr_ncp_20230818T1143415582_b_brw_n18.xml
E:\ISRO PS-Transformer\DataSet\OHRC\ch2_ohr_ncp_20230818T1143415582_d_img_n18\data\calibrated\20230818\ch2_ohr_ncp_20230818T1143415582_d_img_n18.xml
E:\ISRO PS-Transformer\DataSet\OHRC\ch2_ohr_ncp_20230818T1143415582_d_img_n18\geometry\calibrated\20230818\ch2_ohr_ncp_20230818T1143415582_g_grd_n18.xml
E:\ISRO PS-Transformer\DataSet\OHRC\ch2_ohr_nrp_20200229T0938004033_d_img_d32\browse\raw\20200229\ch2_ohr_nrp_20200229T0938004033_b_brw_d32.xml
E:\ISRO PS-Transformer\DataSet\OHRC\ch2_ohr_nrp_20200229T0938004033_d_img_d32\data\raw\20200229\ch2_ohr_nrp_20200229T0938004033_d_img_d32.xml


In [ ]:
import cv2
image = cv2.imread(r'E:\ISRO PS-Transformer\DataSet\OHRC\ch2_ohr_nrp_20200229T0938004033_d_img_d32\data\raw\20200229\ch2_ohr_nrp_20200229T0938004033_d_img_d32.img')

In [ ]:
print("Shape:", image.shape)
print("Data type:", image.dtype)


AttributeError: 'NoneType' object has no attribute 'shape'

In [ ]:
if image is None:
    print("Error loading image.")
    cv2.error("cv2.imread")

Error loading image.


In [ ]:
import imageio

file_path = r'E:\ISRO PS-Transformer\DataSet\OHRC\ch2_ohr_nrp_20200229T0938004033_d_img_d32\data\raw\20200229\ch2_ohr_nrp_20200229T0938004033_d_img_d32.img'

# Open the image using imageio
image = imageio.imread(file_path)

# Now 'image' is a NumPy array containing the image data
print("Image shape:", image.shape)


C:\Users\Sukhvansh Jain\AppData\Local\Temp\ipykernel_30752\3513152778.py:6: DeprecationWarning: Starting with ImageIO v3 the behavior of this function will switch to that of iio.v3.imread. To keep the current behavior (and make this warning disappear) use `import imageio.v2 as imageio` or call `imageio.v2.imread` directly.
  image = imageio.imread(file_path)


RuntimeError: Could not create IO object for reading file E:\ISRO PS-Transformer\DataSet\OHRC\ch2_ohr_nrp_20200229T0938004033_d_img_d32\data\raw\20200229\ch2_ohr_nrp_20200229T0938004033_d_img_d32.img

In [ ]:
import pds4_tools as pds4
import spiceypy as spice

In [ ]:
import requests
import threading
import time

cookies = {"JSESSIONID": "38424b2078f09d20d0d3f1e84614"}
url_prefix = "https://pradan.issdc.gov.in"
proxy_options = {}  # Set proxy options here if needed

def keep_alive():
    """Sends keep-alive requests periodically to maintain session."""
    while True:
        response = requests.get(url_prefix + "/ch2/protected/payload.xhtml",
                                cookies=cookies,
                                proxies=proxy_options)
        time.sleep(600)  # Send keep-alive every 10 minutes

# Start keep-alive thread
keep_alive_thread = threading.Thread(target=keep_alive)
keep_alive_thread.start()

data_file_paths = [
    "/ch2/protected/downloadData/POST_OD/isda_archive/ch2_bundle/cho_bundle/nop/ohr_collection/data/calibrated/20230823/ch2_ohr_ncp_20230823T1647285315_d_img_n18.zip"
]

for i, file in enumerate(data_file_paths):
    print(file)
    response = requests.get(url_prefix + file,
                            cookies=cookies,
                            proxies=proxy_options,
                            stream=True)

    # if response.status_code != 200:
    #     print(f"Error: Limits reached or session expired, terminating without downloading file {i+1}: {file}")
    #     print("You may login again later to download script for the new session and resume downloads.")
    #     keep_alive_thread._stop()  # Stop keep-alive thread
    #     exit(-1)

    with open(file.split("/")[-1], 'wb') as f:
        for chunk in response.iter_content(chunk_size=1024):
            f.write(chunk)

print("Your downloads are complete.")

keep_alive_thread._stop()  # Stop keep-alive thread


/ch2/protected/downloadData/POST_OD/isda_archive/ch2_bundle/cho_bundle/nop/ohr_collection/data/calibrated/20230823/ch2_ohr_ncp_20230823T1647285315_d_img_n18.zip
Your downloads are complete.


AssertionError: 